# CLOAF on a GPU notebook (Colab or Kaggle)

**Colab:** Runtime → Change runtime type → GPU. **Kaggle:** Settings → Accelerator → GPU, and add the data as a Kaggle dataset instead of Drive (change `DATA` below).

Put the licensed data in Google Drive (never in the repo):

```
MyDrive/cloaf_data/
  SMPL_python_v.1.1.0.zip (from smpl.is.tue.mpg.de, as downloaded)
  amass/ACCAD.tar.bz2     (AMASS, "SMPL+H G") -> training
  amass/SFU.tar.bz2       (AMASS, "SMPL+H G") -> test sequences
```

Everything that matters (checkpoints, logs, results) is written to Drive, so a disconnect loses nothing:
reconnect, run the setup cells again and continue.

## Setup (run again after every disconnect)

In [ ]:
!nvidia-smi

In [ ]:
import os
os.chdir('/content')
if not os.path.exists('/content/cloaf_impl'):
    !git clone https://github.com/arhannlp/cloaf_impl.git /content/cloaf_impl
%cd /content/cloaf_impl
!git pull -q
!pip install -q torchdiffeq smplx trimesh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, os, subprocess
DATA = '/content/drive/MyDrive/cloaf_data'
RES = f'{DATA}/results'
for d in ['data/body_models', 'data/amass', f'{DATA}/checkpoints', RES]:
    os.makedirs(d, exist_ok=True)
print(os.listdir(DATA), os.listdir(f'{DATA}/amass'))

# SMPL: official zip -> chumpy-free SMPL_*.pkl
!python scripts/convert_smpl_pkl.py --src {DATA}/SMPL_python_v.1.1.0.zip --out data/body_models/smpl

# AMASS: extract each archive once (a re-run only extracts archives added since, e.g. CMU)
for f in sorted(glob.glob(f'{DATA}/amass/*.tar.bz2')):
    done = f'data/amass/.{os.path.basename(f)}.done'
    if not os.path.exists(done):
        print('extracting', os.path.basename(f), '...')
        subprocess.run(['tar', '-xjf', f, '-C', 'data/amass'], check=True)
        open(done, 'w').close()
print('AMASS folders:', [d for d in os.listdir('data/amass') if not d.startswith('.')])
print('AMASS files:', len(glob.glob('data/amass/**/*.npz', recursive=True)))
assert os.path.exists('data/body_models/smpl/SMPL_NEUTRAL.pkl'), 'SMPL conversion failed: read the messages above'
print('setup OK')

## Step 0 – smoke test (fake body, must print SMOKE TEST OK) and real SMPL check (PASS lines)

In [ ]:
!python tests/smoke_test.py 2>&1 | grep -E 'PASS|FAIL|SKIP|SMOKE'
!python tests/check_smpl.py --smpl data/body_models/smpl 2>&1 | grep -v WARNING

## Step 1 – Jacobian check on the real SMPL (compare with Fig. 5 / B.7; write the numbers down)

In [ ]:
!python scripts/check_linearity.py --smpl data/body_models/smpl --restarts 3 --out {DATA}/results/linearity.npz

## Step 2 – simple target field, no training (hand-target distance must shrink)

In [ ]:
!python scripts/demo_target_field.py --smpl data/body_models/smpl --out {DATA}/results/target_field.npz

## Step 3 – prepare AMASS: train on ACCAD, test on SFU (held out)

In [ ]:
# Folders created by extracting the AMASS archives. To add CMU later (second training
# block), set TRAIN_SETS = ['ACCAD', 'CMU'], re-run the setup cell and this cell, then continue training.
TRAIN_SETS, TEST_SET = ['ACCAD'], 'SFU'
train_roots = [f'data/amass/{t}' for t in TRAIN_SETS if os.path.isdir(f'data/amass/{t}')] or ['data/amass']
test_root = f'data/amass/{TEST_SET}' if os.path.isdir(f'data/amass/{TEST_SET}') else train_roots[0]
ROOTS = ' '.join(train_roots)
!python scripts/prepare_amass.py --amass_root {ROOTS} --out data/amass_smpl_30fps.npz
SEQS = sorted(s for s in glob.glob(f'{test_root}/**/*.npz', recursive=True) if not s.endswith('shape.npz'))
TEST = [SEQS[i] for i in (0, len(SEQS) // 3, 2 * len(SEQS) // 3)]   # 3 different sequences
print('train on', train_roots, '| test on', test_root)
print(*TEST, sep='\n')

## Step 4 – training-free baselines (run while you wait, or before training)

* `point_target`: Supp. C "simple field"; expected to keep the collisions (no non-intersection property).
* `smooth_target`: smooth in space, so touching points move together; our own training-free variant.

In [ ]:
for seq in TEST:
    name = os.path.basename(seq)[:-4]
    for fld in ['point_target', 'smooth_target']:
        out = f'{RES}/{name}_{fld}_all.npz'
        !python scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" --field {fld} --frames 90 --corrupt --mode all --out "{out}" 2>&1 | grep -v WARNING

## Step 5 – train the motion field

Re-run this cell to **continue** training (it resumes from Drive). Each run lasts `--minutes`.
The loss must go down; if it does not after ~20 minutes, stop and check.

In [ ]:
!python -m cloaf.train --smpl data/body_models/smpl --amass data/amass_smpl_30fps.npz \
    --batch 16 --minutes 120 --resume --out {DATA}/checkpoints/field.pt

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_csv(f'{DATA}/checkpoints/field.pt.log.csv')
plt.semilogy(log.iteration, log.loss); plt.xlabel('iteration'); plt.ylabel('loss (avg of 100)'); plt.grid(); plt.show()
log.tail()

## Step 6 – CLOAF with the trained network, both modes (the project extension)

In [ ]:
CKPT = f'{DATA}/checkpoints/field.pt'
for seq in TEST:
    name = os.path.basename(seq)[:-4]
    for mode in ['all', 'colliding_only']:
        out = f'{RES}/{name}_net_{mode}.npz'
        !python scripts/run_on_sequence.py --smpl data/body_models/smpl --seq "{seq}" --field "{CKPT}" --frames 90 --corrupt --mode {mode} --out "{out}" 2>&1 | grep -v WARNING

## Step 7 – summary table (one row per run; also saved as results/summary.csv in Drive)

In [ ]:
import pandas as pd
pd.read_csv(f'{RES}/summary.csv')